In [87]:
import pandas as pd
import numpy as np
# import the functions for cosine distance, euclidean distance, and the correlation distance
from scipy.spatial.distance import cosine, euclidean

#### Import data from a text file (tab delimited)

In [88]:
df_data1 = pd.read_csv("DBbook_train_ratings.tsv", # the location to the data file
                       sep="\t" # for tab delimited documents, use "\t" as the seperator
                        # define the names for the four columns
                       )

print(df_data1.head())
print(df_data1.shape)

   userID  itemID  rate
0    6873    3201     4
1    6873    3098     4
2    6873    4198     4
3    6873    5950     4
4    6873     204     4
(75558, 3)


# TASK 1

1. Please write code to print the total number of ratings, the number of unique users and the number of unique books in this data set. (C)(O)

#### Examining the numbers of ratings, unique users, and unique items

In [89]:
# df_data1["user"].unique() returns the unique values in the "user" column
# the len function returns the length of a list-like object
print("Ratings: %d" % len(df_data1))  #total number of ratings
print("Unique user IDs: %d" % len(df_data1["userID"].unique()))  #number of unique users 
print("Unique items: %d" % len(df_data1["itemID"].unique()))   #number of unique books 

Ratings: 75558
Unique user IDs: 6181
Unique items: 6166


# TASK 2

2. Please write code to create the utility matrix. Each row of this matrix represents a user, and each column represents an item. Print the first 5 rows of matrix. Please write code to print the total number of cells in the utility matrix that are not populated. Please write code to fill these empty cells with 0s. (C)(O)

In [90]:
# #rank items based on rating frequency
# df_item_freq = df_data1.groupby('itemID').count()
# print(df_item_freq)
# df_item_freq.sort_values(by='rate',ascending = False).head()

In [91]:
utility_matrix = df_data1.pivot_table(values="rate", index=["userID"], columns=["itemID"])
print("Shape of the user-item matrix: %d x %d" % utility_matrix.shape)
print(utility_matrix.head())

Shape of the user-item matrix: 6181 x 6166
itemID  1     2     3     5     7     8     9     11    12    13    ...  8157  \
userID                                                              ...         
1        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   
2        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   
3        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   
4        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   
5        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   

itemID  8160  8161  8162  8163  8164  8166  8167  8168  8169  
userID                                                        
1        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4        NaN   NaN   NaN   NaN   NaN   NaN   NaN   4.0   NaN  
5        Na

NaN stands for "Not a Number", which usually mean missing value.

In [92]:
# the iloc method returns a row from a data frame based on a row index
user1_ratings = utility_matrix.iloc[1]
# the isnull() method of a pandas series checks each value to see if it is null (NaN)
print(len(user1_ratings.isnull()))
# the following line selects values user1_ratings that are not null
user1_not_null = user1_ratings[user1_ratings.notnull()]
# print the number of item (non-missing ratings) rated by User 1
len(user1_not_null)

6166


8

In [93]:
missing_values_count = utility_matrix.isnull().sum().sum()
print(f"Total missing values in the utility_matrix are: {missing_values_count}")

Total missing values in the utility_matrix are: 38036488


### Replace missing rating with 0s

Missing values (NaN) usually cannot be used in numerical calculations

In [94]:
utility_matrix = utility_matrix.fillna(0)
utility_matrix.head()

itemID,1,2,3,5,7,8,9,11,12,13,...,8157,8160,8161,8162,8163,8164,8166,8167,8168,8169
userID,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [95]:
print(f"Missing values after filling: {utility_matrix.isna().sum().sum()}")

Missing values after filling: 0


# Task 3

3. Please write code to print the top 5 similar users to userID 5 based on Euclidean distance.  (C)(O)

In [96]:
from scipy.spatial.distance import euclidean

def top_k_users(user_number, k):

    # Removing the active user from the dataset
    df_sim = utility_matrix.loc[utility_matrix.index != user_number].copy()

    # Computing the Euclidean distance for each user
    df_sim["distance"] = df_sim.apply(lambda x: euclidean(utility_matrix.loc[user_number], x), axis=1)

    # Return top k most similar users sorted by distance
    return df_sim.sort_values(by="distance").head(k)["distance"]

# Retrieving top five similar users to User ID 5
top_k_users(5, 5)


userID
3672    12.369317
2151    13.747727
6413    14.142136
6875    14.352700
2436    14.560220
Name: distance, dtype: float64

# Task 4

4. Please write code to print the Euclidean distance between itemID 18 and itemID 1. Please write code to print the Enclidean distance between itemID36 and itemID 1. Write a print statement that tells me between itemID36 and itemID18, which is more similar to itemID 1 and why. For example, you can write a print statement like print(“itemID36 is more similar to itemID 1 because some reason…” ). (C)(O)

In [97]:
from scipy.spatial.distance import euclidean

def item_distance(item1, item2):
    
    return euclidean(utility_matrix[item1], utility_matrix[item2])

# Computing Euclidean distances
distance_18_1 = item_distance(18, 1)
distance_36_1 = item_distance(36, 1)

print(f"Euclidean distance between Item 18 and Item 1 is : {distance_18_1:.4f}")
print(f"Euclidean distance between Item 36 and Item 1 is : {distance_36_1:.4f}")

# To know which item is more similar to Item 1
if distance_36_1 < distance_18_1:
    print("Item 36 is more similar to Item 1 because it has a smaller Euclidean distance.")
else:
    print("Item 18 is more similar to Item 1 because it has a smaller Euclidean distance.")


Euclidean distance between Item 18 and Item 1 is : 29.1890
Euclidean distance between Item 36 and Item 1 is : 40.1248
Item 18 is more similar to Item 1 because it has a smaller Euclidean distance.


# Task 5

5. Please write code to print the top 5 similar items to itemID 8010. (C)(O)

In [98]:
from scipy.spatial.distance import euclidean

def top_k_items(item_number, k):

    # Copying the utility matrix and transposing it so each row represents an item
    df_sim = utility_matrix.transpose().copy()

    # To Remove active item from the dataset
    df_sim = df_sim.loc[df_sim.index != item_number]

    # Computing the Euclidean distance for each item
    df_sim["distance"] = df_sim.apply(lambda x: euclidean(utility_matrix[item_number], x), axis=1)

    # Returning the top k most similar items sorted by distance
    return df_sim.sort_values(by="distance").head(k)["distance"]

# now Retrieving top five similar items to Item 8010
top_k_items(8010, 5)


itemID
3711    127.921851
4559    127.964839
330     129.715072
1311    129.722781
7328    129.761319
Name: distance, dtype: float64

# Task 6

6. Write code to remove books and users with less than 20 rating scores from the utility matrix by copying and maybe modifying the following codes. Write code to print the shape of the dataset. (C)(O)

In [99]:
#To Counting how many times each book (item) has been rated
df_item_fre = df_data1.groupby("itemID").count()

#To Count how many times each user has given a rating
df_user_fre = df_data1.groupby("userID").count()

#Selecting books (items) which are more than 20 ratings
selected_items = df_item_fre[df_item_fre["userID"] > 20].index
dense_matrix = utility_matrix[selected_items]

#Selecting users which have rated more than 20 books
selected_users = df_user_fre[df_user_fre["itemID"] > 20].index
dense_matrix = dense_matrix.loc[selected_users]

#To Print the new shape of the dataset after removing Books and Users with Less Than 20 Ratings
print(f"Shape of dataset after filtering: {dense_matrix.shape}")

Shape of dataset after filtering: (766, 776)


# Task 7

7. Please use the dataset you obtained from task 6 and write code to remove users that haven’t rated itemID8010, and then please write code to print the counts of the different rating scores of this item (hint: use the function value_counts()). Print the shape of the dataset. (C)(O)

In [100]:
# Removing users who haven't rated itemID 8010
dense_matrix = dense_matrix[dense_matrix[8010] > 0]

#Printing the count of different rating scores for item 8010
rating_counts = dense_matrix[8010].value_counts()
print("Rating counts for item 8010:")
print(rating_counts)

# To Print the shape of the dataset after removing users  that haven’t rated itemID8010
print(f"Shape of dataset after filtering users who rated item 8010: {dense_matrix.shape}")


Rating counts for item 8010:
8010
4.0    68
5.0    58
3.0    27
2.0    13
1.0     8
Name: count, dtype: int64
Shape of dataset after filtering users who rated item 8010: (174, 776)


# Task 8

8. Write code to partition the data set you obtained from 7 for validating the performance on predicting rating on itemID 8010. Randomly select 25% of the users as the testing set and the others as the training set. Please print the dimensions of the training set and the testing set. Please write code to print the mean rating of itemID 8010 in the training set and its mean rating in the testing set. (C)(O)

In [101]:
from sklearn.model_selection import train_test_split

# Creating a DataFrame for predictors (we exclude itemID 8010)
df_x = dense_matrix.drop(columns=[8010])
print(f"Predictor dataset shape: {df_x.shape}")

# Creating a Series for the outcome (ratings for itemID 8010)
df_y = dense_matrix[[8010]]
print(f"Target dataset shape: {df_y.shape}")

# Split the data into 75% training and 25% testing
train_x, test_x, train_y, test_y = train_test_split(df_x, df_y, test_size=0.25, random_state=42)

# Converting to DataFrame format
df_train_x = pd.DataFrame(train_x, columns=df_x.columns)
df_test_x = pd.DataFrame(test_x, columns=df_x.columns)
df_train_y = pd.DataFrame(train_y, columns=[8010])
df_test_y = pd.DataFrame(test_y, columns=[8010])

#Printing shapes of training and testing sets
print(f"Train X: {df_train_x.shape}, Train Y: {df_train_y.shape}")
print(f"Test X: {df_test_x.shape}, Test Y: {df_test_y.shape}")

# now Compute mean rating of itemID 8010 in both sets
train_mean_rating = df_train_y[8010].mean()
test_mean_rating = df_test_y[8010].mean()
# now printing mean rating of itemID 8010 in both sets
print(f"Mean rating of itemID 8010 in Training Set: {train_mean_rating:.4f}")
print(f"Mean rating of itemID 8010 in Testing Set: {test_mean_rating:.4f}")


Predictor dataset shape: (174, 775)
Target dataset shape: (174, 1)
Train X: (130, 775), Train Y: (130, 1)
Test X: (44, 775), Test Y: (44, 1)
Mean rating of itemID 8010 in Training Set: 3.9769
Mean rating of itemID 8010 in Testing Set: 3.6364


# Task 9

9. Use the training and test dataset obtained in 8 and write code to 1) print the userID of the the user in the 4th row (not userID5) in the test dataset, and 2) predict this user’s rating of itemID 8010 based on the top 10 similar users in the training dataset, and print the user’s predicted rating and the actual rating of the book.  (C)(O)

In [102]:
from scipy.spatial.distance import euclidean

# To Get the userID of the 4th row in the test dataset (we exclude userID 5)
user_4th = df_test_x.index[3]  # user in the 4th row of test set
print(f"User ID in the 4th row of test dataset (not userID5): {user_4th}")

#Predicting the rating of itemID 8010 for this user based on top 10 similar users in training data
#Number of similar users to consider
k = 10  
def user_based_predict(user_number):

    # Retrieving the top k similar users
    df_sim = df_train_x.copy()
    # to Compute Euclidean distance between the target user and all training users
    df_sim["distance"] = df_sim.apply(lambda x: euclidean(df_test_x.loc[user_number], x), axis=1)
    # now Selecting top k most similar users
    df_sim_users = df_sim.loc[df_sim.sort_values(by="distance").head(k).index]
    # next to Compute weighted sum of ratings for itemID 8010
    df_sim_users["weighted_d"] = df_sim_users.apply(lambda x: x["distance"] * df_train_y.loc[x.name, 8010], axis=1)
    # now to Predict rating using weighted average
    predicted_rating = df_sim_users["weighted_d"].sum() / df_sim_users["distance"].sum()
    
    return predicted_rating

# to Compute the predicted rating for user in 4th row of test dataset
predicted_rating = user_based_predict(user_4th)
# to Get the actual rating from test dataset
actual_rating = df_test_y.loc[user_4th, 8010]

#Printing the results
print(f"Predicted rating for User {user_4th} on itemID 8010: {predicted_rating:.4f}")
print(f"Actual rating given by User {user_4th} on itemID 8010: {actual_rating:.4f}")

User ID in the 4th row of test dataset (not userID5): 5908
Predicted rating for User 5908 on itemID 8010: 3.7975
Actual rating given by User 5908 on itemID 8010: 5.0000


# Task 10


## 1) How is “collaborative filtering” different from “content filtering” in recommender systems?
Collaborative Filtering and Content-Based Filtering are the two major approaches used in recommendation systems for suggesting items to users.

*Collaborative Filtering*
Generally, This method recommends items based on user behavior and preferences.
It assumes that users who liked similar items in the past will like similar items in the future.
It does not require any knowledge about the item’s features.
so, It is widely used in platforms like Netflix and Amazon, where recommendations are based on what similar users have watched or purchased.

Advantages: It Can recommend diverse items beyond a user’s past choices and also it does not need detailed information about items.
Disadvantages: It Struggles with the cold start problem (i.e., new users or new items with no prior interactions).
basically it Requires a large amount of data to make accurate recommendations.

*Content-Based Filtering*
This method recommends items based on their characteristics and how well they match a user’s past preferences.
If a user has watched many romance movies, the system will also recommend more romance movies.
It relies on detailed item information, such as genre, author, director, keywords, etc.
Advantages: It Works well for new users because it only needs their past preferences and does not rely on other users’ behavior, making it more personalized.
Disadvantages: It is Limited to recommending items similar to what the user has already engaged with so. it requires well-structured data about item attributes.

## 2) What are the differences between neighborhood methods (memory-based) and latent factor models (model-based)?

we know Collaborative filtering helps recommend things (like movies, books or items) based on what similar users have liked. There are two main ways to do this:

*1. Neighborhood Methods (Memory-Based Collaborative Filtering)*
This method Looks at users or items that are similar to each other and Uses methods like Euclidean distance or cosine similarity to find similarities. for example in User-based approach, If two people have liked the same movies before, they will get similar recommendations. and for Item-based approach, If two movies are rated similarly by the same users, they are considered similar.
Advantages: Easy to understand and it Works well when there is enough data.
Disadvantages: Its Slow on large datasets and also Cold start problem – If a new user or item has no data, it struggles to make recommendations.

*2. Latent Factor Models (Model-Based Collaborative Filtering)*
This Uses advanced techniques like Matrix Factorization (SVD, PCA) or Deep Learning to discover hidden patterns.
so, Instead of directly comparing users or items, it finds underlying “factors” that influence preferences.
for example, Instead of saying “Alice and Sarah both like action movies,” it discovers that Alice likes movies with explosions, fast cars and fights, so it recommends movies with those features.
Advantages: It Works well even when there is very little user data (sparse data) and also it Can handle huge datasets efficiently (Netflix, Amazon and Spotify use this method).
Disadvantages: Its Hard to explain why a recommendation was made. and also it Needs training and computing power, unlike neighborhood methods that work out-of-the-box.




## 3) Explain SVD and matrix factorization in your own words (max 10 sentences).

Singular Value Decomposition (SVD) and Matrix Factorization are the techniques used in recommender systems to break down large datasets into smaller, more meaningful parts.

Imagine we have a huge table (matrix) where rows are users, columns are movies and cells contain ratings.
so, This table is mostly empty because users haven’t rated every movie. Now, SVD helps reduce the size of this matrix while keeping important patterns, kind of like summarizing a big book into key points. and It breaks the matrix into three smaller matrices: user preferences, movie features, and importance scores. so, By keeping only the most important features, it removes noise (random ratings) and helps predict missing ratings.
For example, instead of storing 1,000,000 ratings, SVD might find that “Action vs. Romance” and “Popular vs. Niche” are the most important factors.
If a user likes fast-paced action movies, the system can recommend similar ones even if they haven’t rated them before.
so, This is why Netflix, Spotify and Amazon use matrix factorization to suggest movies, songs, and products.
and the disadvantage is, It’s harder to explain recommendations because the model discovers hidden patterns we don’t directly see.
Overall, SVD is like a smart librarian who understands our taste and recommends books based on what others with similar interests enjoyed.

# Reference Code Chunks


#### Examine the distribution of the number of ratings for each item

In [73]:
# %matplotlib inline
# # count the non-zero elements for each column (item)
# rating_counts_by_item = sparse_matrix.apply(np.count_nonzero, axis=0) # strange. Usually axis =0 represent row, but here we count
#                                                                      # non-zero values in a column
# print(rating_counts_by_item)
# # plot a histogram
# import matplotlib.pyplot as plt
# plt.hist(rating_counts_by_item.values, bins=100)
# plt.ylabel("Number of items")
# plt.xlabel("Number of ratings")
# plt.show()

The histogram above shows a long-tail distribution:
<li>The majority of items were not rated by many users (the fisrt a few bins from the left)</li>
<li>Only a few most popular items have a large number of ratings (bins from the right)</li>

In [74]:
# # count the non-zero elements for each row (user)
# rating_counts_by_user = sparse_matrix.apply(np.count_nonzero, axis=1)
# print("Average # of ratings per user:", sum(rating_counts_by_user)/float(len(rating_counts_by_user)))

# # plot a histogram
# import matplotlib.pyplot as plt
# plt.hist(rating_counts_by_user.values, bins=100)
# plt.ylabel("Number of users")
# plt.xlabel("Number of ratings")
# plt.show()

The histogram above shows a long-tail distribution:
<li>The majority of user only rated a few (the fisrt a few bins from the left)</li>
<li>Only a few user have rated a large number of items (bins from the right)</li>

### Calculating Similarity

<b>Cosine Distance</b> <br>
http://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.spatial.distance.cosine.html#scipy.spatial.distance.cosine
    

In [61]:
# # a user's cosine distance to itself (virtually 0)
# cosine(sparse_matrix.loc[1], sparse_matrix.loc[1])

0

In [62]:
# cosine(sparse_matrix.loc[1], sparse_matrix.loc[2])

0.833069016131298

In [63]:
# cosine(sparse_matrix[1], sparse_matrix[2])

0.5976178217003905

<b>Euclidean Distance</b><br>
http://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.spatial.distance.euclidean.html#scipy.spatial.distance.euclidean

In [64]:
# # a user's Euclidean distance to itself (0)
# euclidean(sparse_matrix.loc[1], sparse_matrix.loc[1])

0.0

In [65]:
# euclidean(sparse_matrix.loc[1], sparse_matrix.loc[2])

65.25335240430181

In [66]:
# euclidean(sparse_matrix[1], sparse_matrix[2])

77.72387020729218

### Finding the K most similar items/user to a given item/user

#### Using Euclidean distance

In [67]:
# # define a functions, which takes the given item number (integer) as the input and returns the top K similar items (in a data frame)
# def top_k_items(item_number, k):
#     # copy the dense matrix and transpose it so each row represents an item
#     df_sim = sparse_matrix.transpose()
#     # remove the active item 
#     df_sim = df_sim.loc[df_sim.index != item_number]
#     # calculate the distance between the given item for each row (apply the function to each row if axis = 1)
#     df_sim["distance"] = df_sim.apply(lambda x: euclidean(sparse_matrix[item_number], x), axis=1)
#     # return the top k from the sorted distances
#     return df_sim.sort_values(by="distance").head(k)["distance"]   

In [68]:
# # retrieve top five similar items to Item 1
# top_k_items(1, 5)

item
121    63.529521
405    65.764732
117    65.840717
151    66.835619
118    67.535176
Name: distance, dtype: float64

In [69]:
# def top_k_users(user_number, k):
#     # no need to transpose the matrix this time because the rows already represent users
#     # remove the active user
#     df_sim = sparse_matrix.loc[sparse_matrix.index != user_number]
#     # calculate the distance for between the given user and each row
#     df_sim["distance"] = df_sim.apply(lambda x: euclidean(sparse_matrix.loc[user_number], x), axis=1)
#     # return the top k from the sorted distances
#     return df_sim.sort_values(by="distance").head(k)["distance"] 

In [70]:
# # retrieve top five similar users to User 3
# top_k_users(3, 5)

C:\Users\jliu2188\AppData\Local\Temp\ipykernel_3680\1378599790.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sim["distance"] = df_sim.apply(lambda x: euclidean(sparse_matrix.loc[user_number], x), axis=1)


user
317    20.952327
656    21.354157
920    21.424285
335    21.424285
155    21.840330
Name: distance, dtype: float64

#### K Nearest Neighbors Algorithm

In [71]:
from sklearn.neighbors import NearestNeighbors
# create an instance of the learner and specify it to use Euclidean distance
nbrs = NearestNeighbors(n_neighbors=5, metric="euclidean")
# fit the learner using all user ratings except for User 3
nbrs.fit(sparse_matrix.loc[sparse_matrix.index != 3])
user3= sparse_matrix.loc[3].values.reshape((-1,len(sparse_matrix.loc[3])))
#user3= dense_matrix.loc[3]
print(user3)
# the learner returns the distances and locations of the 5 nearest neighbors of User 3
distances, locs = nbrs.kneighbors(user3, 5)
print(distances, locs)
# retrieve these neighbors from the user-item matrix based on the locations
# the ravel method returns a flattened list. In the code below, it converts a 2D array with one row to a 1D array
sim_users = sparse_matrix.loc[sparse_matrix.index != 3].iloc[locs.ravel()].index
# print the user indexes and the distances
for sim_user, dist in zip(sim_users, distances.ravel()):
    print(sim_user, dist)

[[0. 0. 0. ... 0. 0. 0.]]
[[20.95232684 21.3541565  21.42428529 21.42428529 21.84032967]] [[315 654 333 918 153]]
317 20.952326839756964
656 21.354156504062622
335 21.42428528562855
920 21.42428528562855
155 21.840329667841555


### Partitioning the data for cross validation

70% of the users are used for training; 30% of the users are used for testing.

#### We will conduct cross validation to predict user ratings for Item 151

In [72]:
sparse_matrix[151].value_counts()

0.0    617
4.0    108
3.0     85
5.0     81
2.0     40
1.0     12
Name: 151, dtype: int64

### Partition for cross validation

In [73]:
from sklearn.model_selection import train_test_split

# create a data frame for the predictors
df_x = sparse_matrix[[col for col in sparse_matrix.columns if col != 151]]
print(df_x.shape)

# create a series for the outcome
df_y = sparse_matrix[[151]]
print(df_y.shape)

train_x, test_x, train_y, test_y = train_test_split(df_x, df_y, test_size=0.3, random_state=0)
df_train_x = pd.DataFrame(train_x, columns=df_x.columns)
df_test_x = pd.DataFrame(test_x, columns=df_x.columns)
df_train_y = pd.DataFrame(train_y, columns=[151])
df_test_y = pd.DataFrame(test_y, columns=[151])
print("shapes")
print(df_train_x.shape)
print(df_test_x.shape)
print(df_train_y.shape)
print(df_test_y.shape)
print() 
print("class counts")
print(df_train_y[151].value_counts())
print(df_test_y[151].value_counts())

(943, 1681)
(943, 1)
shapes
(660, 1681)
(283, 1681)
(660, 1)
(283, 1)

class counts
0.0    416
4.0     78
3.0     65
5.0     63
2.0     28
1.0     10
Name: 151, dtype: int64
0.0    201
4.0     30
3.0     20
5.0     18
2.0     12
1.0      2
Name: 151, dtype: int64


### Creat a function for user-based prediction

In [74]:
# specify the number of similar users to retrieve
k = 5

def user_based_predict(user_number):
    # retrieve the top k similar users
    # copy from all the training predictors
    df_sim = df_train_x.copy()
    # for each user, calculate the distance between this user and the active user
    df_sim["distance"] = df_sim.apply(lambda x: euclidean(df_test_x.loc[user_number], x), axis=1)
    # create a new data frame to store the top k similar users
    df_sim_users = df_sim.loc[df_sim.sort_values(by="distance").head(k).index]
    # calculate these similar users' rating on 151, weighted by distance
    df_sim_users["weighed_d"] = list(map(lambda x: df_sim_users.loc[x]["distance"]*df_train_y.loc[x][151], df_sim_users.index))
    predicted = df_sim_users["weighed_d"].sum()/df_sim_users["distance"].sum()
    return predicted

#### Predict a single user

In [75]:
print(df_test_x.head())

item  1     2     3     4     5     6     7     8     9     10    ...  1673  \
user                                                              ...         
346    0.0   5.0   3.0   4.0   0.0   0.0   2.0   0.0   0.0   0.0  ...   0.0   
877    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   0.0   
559    0.0   0.0   0.0   4.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   0.0   
668    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   0.0   
237    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   4.0   0.0  ...   0.0   

item  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user                                                        
346    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
877    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
559    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
668    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
237    0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  

[5 rows x 1681 col

In [76]:
uid = df_test_x.index[5] 
print(uid)
print("This user's rating on other items: ", df_test_x.loc[uid])
print()
print("Predicted rating on Item 151:", user_based_predict(uid))
print("True rating on Item 151:     ", df_test_y.loc[uid][151])

262
This user's rating on other items:  item
1       3.0
2       0.0
3       0.0
4       0.0
5       0.0
       ... 
1678    0.0
1679    0.0
1680    0.0
1681    0.0
1682    0.0
Name: 262, Length: 1681, dtype: float64

Predicted rating on Item 151: 1.000620934073575
True rating on Item 151:      0.0


#### Predict all testing data

In [77]:
pred_151 = list(map(user_based_predict, df_test_x.index))
print("Number of ratings predicted: %d" % len(pred_151))

Number of ratings predicted: 283


#### Calculate prediction performance metrics

In [78]:
from sklearn.metrics import mean_absolute_error
print("Mean Absolute Error: ", mean_absolute_error(pred_151, df_test_y[151]))

Mean Absolute Error:  0.9462696498329264


You expect to make a error of 0.95 out of the 5-point rating scale. Not too bad. But not too accurate, either.

#### Model based collaborative filtering - matrix factorization (via SVD)

The measure we use is now RMSE

In [79]:
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten() 
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return sqrt(mean_squared_error(prediction, ground_truth))

The code for SVD is below. However, we usually do not use it for recommendation in real practice

In [86]:
import scipy.sparse as sp 
from scipy.sparse.linalg import svds 
#get SVD components from train matrix. Choose k. 
u, s, vt = svds(sparse_matrix.to_numpy(), k = 20) # k is number of elements in feature vectors
s_diag_matrix=np.diag(s) 
X_pred = np.dot(np.dot(u, s_diag_matrix), vt) 
print('model based CF results: ', rmse(X_pred, sparse_matrix.values))

<class 'pandas.core.frame.DataFrame'>
model based CF results:  2.131802113039339


#### Model based collaborative filtering - Stochastic gradient descent (SGD)

Let me first show you how to create a training vs. validation dataset

In [87]:
# This creats a validation dataset by selecting rows (users) that have 50 or more ratings, then randomly select 25 of those ratings
#for validation set, but set those values to 0 in the training set.

def train_test_split(ratings):
    
    validation = np.zeros(ratings.shape)
    train = ratings.copy() #don't do train=ratings, other wise, ratings becomes empty
    for user in np.arange(ratings.shape[0]):
        if len(ratings[user,:].nonzero()[0])>=50:# change this based on sparsity of your user-item matrix
            val_ratings = np.random.choice(ratings[user, :].nonzero()[0], 
                                        size=25, #tweak this
                                        replace=False)
            train[user, val_ratings] = 0
            validation[user, val_ratings] = ratings[user, val_ratings]
    print(validation.shape)
    print(train.shape)
    return train, validation

train, val = train_test_split(sparse_matrix.values)

(943, 1682)
(943, 1682)


Dot product of two vectors results in one rating. Doc prodct of two matrix gives a rating table.

In [88]:
#P is latent user feature matrix
#Q is latent item feature matrix
def prediction(P,Q):
    return np.dot(P.T,Q)

Initialize the algorithm parameters

In [89]:
lmbda = 0.4 # Regularization parameter
k = 20 #Number of feastures; tweak this parameter 
m, n = train.shape  # Number of users and items
print(m, n)
n_epochs = 50  # Number of epochs
alpha=0.01  # Learning rate
P = np.random.rand(k,m) # initial user feature matrix with random numbers
Q = np.random.rand(k,n) # initial movie feature matrix with random numbers
print(P.shape)

943 1682
(20, 943)


SGD learning

In [ ]:
train_errors = []
val_errors = []

#Only consider items with ratings 
users,items = train.nonzero()      
for epoch in range(n_epochs):
    print(epoch)
    for u, i in zip(users,items):
        e = train[u, i] - prediction(P[:,u],Q[:,i])  # Calculate error for gradient update
        P[:,u] += alpha * ( e * Q[:,i] - lmbda * P[:,u]) # Update latent user feature matrix
        Q[:,i] += alpha * ( e * P[:,u] - lmbda * Q[:,i])  # Update latent item feature matrix
    
    train_rmse = rmse(prediction(P,Q),train)
    val_rmse = rmse(prediction(P,Q),val) 
    train_errors.append(train_rmse)
    print(train_errors)
    val_errors.append(val_rmse)

0
[1.4555107166392782]
1
[1.4555107166392782, 1.177654107469178]
2
[1.4555107166392782, 1.177654107469178, 1.1240104052202116]
3
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552]
4
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.0870489064759563]
5
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.0870489064759563, 1.0786561159050951]
6
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.0870489064759563, 1.0786561159050951, 1.072926333371748]
7
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.0870489064759563, 1.0786561159050951, 1.072926333371748, 1.068806987362884]
8
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.0870489064759563, 1.0786561159050951, 1.072926333371748, 1.068806987362884, 1.0657296590514027]
9
[1.4555107166392782, 1.177654107469178, 1.1240104052202116, 1.1003081865259552, 1.08

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.plot(range(n_epochs), train_errors, marker='o', label='Training Data');
plt.plot(range(n_epochs), val_errors, marker='v', label='Validation Data');
plt.xlabel('Number of Epochs');
plt.ylabel('RMSE');
plt.legend()
plt.grid()
plt.show()

Model evaluation:

In [ ]:
SGD_prediction=prediction(P,Q)
estimation= SGD_prediction[val.nonzero()]
ground_truth = val[val.nonzero()]
print(rmse(estimation, ground_truth))
results=pd.DataFrame({'prediction':estimation, 'actual rating':ground_truth})
results.head()